Import Libraries

In [3]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

Load Dataset

In [4]:
df = pd.read_csv(
    "../data/processed/social_media_enriched_sales.csv"
)

df["Date"] = pd.to_datetime(df["Date"])

print(df.shape)

df.head()

(6435, 30)


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,...,Festival_Month,Festival_Impact_Score,Pre_Festival_Flag,Post_Festival_Flag,Trend_Score,Mention_Count,Engagement_Rate,Sentiment_Score,Influencer_Promotion_Flag,Viral_Campaign_Flag
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,2,...,0,21,0,0,30.12,5908,6.40,0.105,0,0
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106,2010,2,...,2,34,1,0,56.02,6544,11.93,0.225,0,0
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106,2010,2,...,2,31,0,1,59.49,10060,11.47,0.994,1,0
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106,2010,2,...,3,35,1,0,71.22,11999,15.74,0.818,1,0
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106,2010,3,...,3,34,0,1,64.28,9335,14.37,0.632,1,0


Sort Data

In [5]:
df = df.sort_values(
    ["Store", "Date"]
)

df.reset_index(
    drop=True,
    inplace=True
)

### Lag Features

Create historical sales features.

In [6]:
df["Lag_1"] = df.groupby(
    "Store"
)["Weekly_Sales"].shift(1)

df["Lag_4"] = df.groupby(
    "Store"
)["Weekly_Sales"].shift(4)

df["Lag_12"] = df.groupby(
    "Store"
)["Weekly_Sales"].shift(12)

Rolling Statistics

In [7]:
df["Rolling_Mean_4"] = (

    df.groupby("Store")
      ["Weekly_Sales"]
      .transform(
          lambda x: x.rolling(4).mean()
      )

)

df["Rolling_Std_4"] = (

    df.groupby("Store")
      ["Weekly_Sales"]
      .transform(
          lambda x: x.rolling(4).std()
      )

)

Moving Average

In [8]:
df["Moving_Average_8"] = (

    df.groupby("Store")
      ["Weekly_Sales"]
      .transform(
          lambda x: x.rolling(8).mean()
      )

)

### Interaction Features

Festival × Trend

In [9]:
df["Festival_Trend"] = (

    df["Festival_Score"]

    *

    df["Trend_Score"]

)

Festival × Sentiment

In [10]:
df["Festival_Sentiment"] = (

    df["Festival_Score"]

    *

    df["Sentiment_Score"]

)

Trend × Engagement

In [11]:
df["Trend_Engagement"] = (

    df["Trend_Score"]

    *

    df["Engagement_Rate"]

)

Sales Growth

In [12]:
df["Sales_Growth"] = (

    df.groupby("Store")
      ["Weekly_Sales"]
      .pct_change()

)

Store Average Sales

In [13]:
store_avg = (

    df.groupby("Store")
      ["Weekly_Sales"]
      .transform("mean")

)

df["Store_Avg_Sales"] = store_avg

Store Sales Rank

In [14]:
store_rank = (

    df.groupby("Store")
      ["Weekly_Sales"]
      .transform("mean")
      .rank()

)

df["Store_Sales_Rank"] = store_rank

### Fill NaNs Created by Lags

Lag features create missing values at the start.

In [15]:
df.fillna(
    method="bfill",
    inplace=True
)

In [16]:
df.isnull().sum().sum()

np.int64(0)

Check New Features

In [17]:
new_features = [

    "Lag_1",
    "Lag_4",
    "Lag_12",

    "Rolling_Mean_4",
    "Rolling_Std_4",

    "Moving_Average_8",

    "Festival_Trend",
    "Festival_Sentiment",
    "Trend_Engagement",

    "Sales_Growth",

    "Store_Avg_Sales",
    "Store_Sales_Rank"

]

df[new_features].head()

,Lag_1,Lag_4,Lag_12,Rolling_Mean_4,Rolling_Std_4,Moving_Average_8,Festival_Trend,Festival_Sentiment,Trend_Engagement,Sales_Growth,Store_Avg_Sales,Store_Sales_Rank
0,1643690.90,1643690.9,1643690.9,1576836.025,112353.415114,1522329.76,0.00,0.000,192.7680,-0.001055,1.555264e+06,5220.0
1,1643690.90,1643690.9,1643690.9,1576836.025,112353.415114,1522329.76,336.12,1.350,668.3186,-0.001055,1.555264e+06,5220.0
2,1641957.44,1643690.9,1643690.9,1576836.025,112353.415114,1522329.76,356.94,5.964,682.3503,-0.018264,1.555264e+06,5220.0
3,1611968.17,1643690.9,1643690.9,1576836.025,112353.415114,1522329.76,569.76,6.544,1121.0028,-0.125462,1.555264e+06,5220.0
4,1409727.59,1643690.9,1643690.9,1554614.970,103135.002548,1522329.76,514.24,5.056,923.7036,0.102913,1.555264e+06,5220.0


Correlation with Target

In [18]:
corr = (

    df.select_dtypes(
        include=np.number
    )

    .corr()["Weekly_Sales"]

    .sort_values(
        ascending=False
    )

)

corr.head(20)

Weekly_Sales                 1.000000
Rolling_Mean_4               0.976341
Moving_Average_8             0.965696
Store_Avg_Sales              0.957818
Lag_1                        0.951759
Store_Sales_Rank             0.943476
Lag_4                        0.938001
Lag_12                       0.915683
Rolling_Std_4                0.444479
Trend_Engagement             0.331182
Trend_Score                  0.330053
Engagement_Rate              0.324820
Mention_Count                0.277283
Viral_Campaign_Flag          0.239202
Sales_Growth                 0.152606
Influencer_Promotion_Flag    0.148377
Festival_Trend               0.135235
Month                        0.076143
Week                         0.074211
Quarter                      0.063363
Name: Weekly_Sales, dtype: float64

Save Final Feature Dataset

In [19]:
df.to_csv(
    "../data/processed/final_featured_sales.csv",
    index=False
)

print(
    "Final feature engineered dataset saved!"
)

Final feature engineered dataset saved!
